<a href="https://colab.research.google.com/github/jaysulk/GENERIC-FNO/blob/main/GENERIC_FNO_Benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#!/usr/bin/env python3
# ============================================================================
# GENERIC-FNO COMPUTE & MEMORY BENCHMARK -- ONE SELF-CONTAINED COLAB CELL.
# Part A = 2D model classes + generators (verbatim from the model module);
# Part C = times FNO vs EP-FNO vs GENERIC-FNO (params, forward, train step,
# peak GPU memory) at the paper's 128^2 config and prints overhead multipliers
# + a LaTeX table for Appendix E. Set BM_BATCH lower if you hit OOM.
# ============================================================================
#!/usr/bin/env python3
"""
GENERIC-FNO Benchmark — 2D
==========================
Fourier Neural Operator with GENERIC thermodynamic structure, in 2D.

Architecture (unchanged from 1D, lifted to 2D):
  - E-net (FNO2d → scalar): learns energy functional E[u]
  - S-net (FNO2d → scalar): learns entropy functional S[u]
  - L(kx,ky): anti-Hermitian diagonal operator (reversible dynamics)
  - M(kx,ky): Hermitian PSD diagonal operator (dissipative dynamics)
  - Dynamics: du/dt = L·δE/δu + M·δS/δu
  - Hard projection: energy conservation + entropy non-decrease

Compares: FNO (vanilla), EP-FNO (energy penalty), GENERIC-FNO (ours)
Tests on: Heat, Wave, Burgers  (all 2D scalar fields)

Default resolution NX=128 (i.e. 128x128 grids). Adjust in run_benchmark().

Key insight: dynamics are CONSTRUCTED from E,S — no bypass possible.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pickle
import time
import math
from collections import defaultdict


# ============================================================================
# Data Generation — 2D PDEs (spectral)
# ============================================================================

def _random_field_2d(nx, ny, max_mode, n_modes, amp_scale=0.5, device='cpu'):
    """Build a random smooth 2D field as a sum of sine modes."""
    x = torch.linspace(0, 2*math.pi, nx+1, device=device)[:-1]
    y = torch.linspace(0, 2*math.pi, ny+1, device=device)[:-1]
    X, Y = torch.meshgrid(x, y, indexing='ij')
    u = torch.zeros(nx, ny, device=device)
    for _ in range(n_modes):
        kx = torch.randint(1, max_mode, (1,)).item()
        ky = torch.randint(1, max_mode, (1,)).item()
        amp = torch.randn(1).item() * amp_scale
        phase = torch.rand(1).item() * 2 * math.pi
        u += amp * torch.sin(kx * X + ky * Y + phase)
    return u


def generate_heat_data_2d(n_samples=150, nx=128, nt=15, dt=0.005, nu=0.02, device='cpu'):
    """2D heat: du/dt = nu*(uxx+uyy). Purely dissipative. Exact in spectral space."""
    kx = torch.fft.fftfreq(nx, d=1.0/nx).to(device)
    ky = torch.fft.rfftfreq(nx, d=1.0/nx).to(device)
    KX, KY = torch.meshgrid(kx, ky, indexing='ij')
    k_sq = KX**2 + KY**2
    decay = torch.exp(-nu * k_sq * dt)

    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(3, 7, (1,)).item()
        u0 = _random_field_2d(nx, nx, nx//8, n_modes, device=device)
        u_hat = torch.fft.rfft2(u0)
        traj = [u0.clone()]
        for t in range(nt):
            u_hat = u_hat * decay
            traj.append(torch.fft.irfft2(u_hat, s=(nx, nx)))
        traj = torch.stack(traj, dim=0)  # (nt+1, nx, nx)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])
    return torch.stack(data_in), torch.stack(data_out), 'heat'


def generate_wave_data_2d(n_samples=150, nx=128, nt=15, dt=0.005, c=1.0, device='cpu'):
    """2D wave: u_tt = c²(uxx+uyy). Reversible. Track u-component, exact spectral rotation."""
    kx = torch.fft.fftfreq(nx, d=1.0/nx).to(device)
    ky = torch.fft.rfftfreq(nx, d=1.0/nx).to(device)
    KX, KY = torch.meshgrid(kx, ky, indexing='ij')
    kmag = torch.sqrt(KX**2 + KY**2)
    omega = c * kmag

    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(3, 7, (1,)).item()
        u0 = _random_field_2d(nx, nx, nx//8, n_modes, device=device)
        v0 = _random_field_2d(nx, nx, nx//8, n_modes, amp_scale=0.3, device=device)
        u_hat = torch.fft.rfft2(u0)
        v_hat = torch.fft.rfft2(v0)
        traj = [u0.clone()]
        for t in range(nt):
            cos_w = torch.cos(omega * dt)
            sin_w = torch.sin(omega * dt)
            # rotate (u, v) preserving energy; guard omega=0 mode
            safe_omega = torch.where(omega > 1e-8, omega, torch.ones_like(omega))
            u_new = cos_w * u_hat + (sin_w / safe_omega) * v_hat
            v_new = -safe_omega * sin_w * u_hat + cos_w * v_hat
            u_hat, v_hat = u_new, v_new
            traj.append(torch.fft.irfft2(u_hat, s=(nx, nx)))
        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])
    return torch.stack(data_in), torch.stack(data_out), 'wave'


def generate_advection_data_2d(n_samples=150, nx=128, nt=15, dt=0.005, c=1.0,
                               max_mode=6, device='cpu'):
    """2D linear advection: u_t + c(u_x+u_y) = 0. Reversible, Markovian in u,
    conserves 0.5<u^2> exactly. Clean fully-observed reversible scalar test
    => a thermodynamically-consistent operator should drive M -> 0.
    Band-limited to max_mode (fixed, NOT nx//8) so the content stays within the
    operator's mode range and the per-step phase rotation is small -- otherwise
    high-k transport aliases and even a plain FNO fails (the operators only span
    the lowest modes_op modes)."""
    kx = torch.fft.fftfreq(nx, d=1.0/nx).to(device)
    ky = torch.fft.rfftfreq(nx, d=1.0/nx).to(device)
    KX, KY = torch.meshgrid(kx, ky, indexing='ij')
    phase = torch.exp(-1j * c * (KX + KY) * dt)
    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(3, 7, (1,)).item()
        u0 = _random_field_2d(nx, nx, max_mode, n_modes, device=device)
        u_hat = torch.fft.rfft2(u0)
        traj = [u0.clone()]
        for t in range(nt):
            u_hat = u_hat * phase
            traj.append(torch.fft.irfft2(u_hat, s=(nx, nx)))
        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])
    return torch.stack(data_in), torch.stack(data_out), 'advection'


def generate_burgers_data_2d(n_samples=150, nx=128, nt=15, dt=0.002, nu=0.02, device='cpu'):
    """2D scalar Burgers: u_t + u(u_x+u_y) = nu*(uxx+uyy). Mixed rev+diss.
    Semi-implicit: diffusion in spectral, advection explicit."""
    kx = torch.fft.fftfreq(nx, d=1.0/nx).to(device)
    ky = torch.fft.rfftfreq(nx, d=1.0/nx).to(device)
    KX, KY = torch.meshgrid(kx, ky, indexing='ij')
    k_sq = KX**2 + KY**2

    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(2, 5, (1,)).item()
        u = _random_field_2d(nx, nx, 5, n_modes, amp_scale=0.3, device=device)
        traj = [u.clone()]
        for t in range(nt):
            u_hat = torch.fft.rfft2(u)
            # implicit diffusion
            u_hat = u_hat / (1 + nu * k_sq * dt)
            u = torch.fft.irfft2(u_hat, s=(nx, nx))
            # explicit advection (spectral derivatives)
            ux = torch.fft.irfft2(1j * KX * torch.fft.rfft2(u), s=(nx, nx))
            uy = torch.fft.irfft2(1j * KY * torch.fft.rfft2(u), s=(nx, nx))
            u = u - dt * u * (ux + uy)
            traj.append(u.clone())
        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])
    return torch.stack(data_in), torch.stack(data_out), 'burgers'


# ============================================================================
# Building Blocks — 2D
# ============================================================================

class SpectralConv2d(nn.Module):
    """Standard FNO spectral convolution (2D). Two corners for +/- kx."""
    def __init__(self, in_ch, out_ch, modes1, modes2):
        super().__init__()
        self.modes1 = modes1  # kx modes (keep both +/- corners)
        self.modes2 = modes2  # ky modes (non-negative only, rfft)
        scale = 1.0 / (in_ch * out_ch)
        self.W1 = nn.Parameter(scale * torch.randn(out_ch, in_ch, modes1, modes2, dtype=torch.cfloat))
        self.W2 = nn.Parameter(scale * torch.randn(out_ch, in_ch, modes1, modes2, dtype=torch.cfloat))

    def forward(self, x):
        B, C, H, W = x.shape
        x_hat = torch.fft.rfft2(x, dim=(-2, -1))  # (B, C, H, W//2+1)
        out_hat = torch.zeros(B, self.W1.shape[0], H, W // 2 + 1,
                              dtype=torch.cfloat, device=x.device)
        m1 = min(self.modes1, H // 2)
        m2 = min(self.modes2, W // 2 + 1)
        # top-left corner (positive kx)
        out_hat[:, :, :m1, :m2] = torch.einsum(
            'bixy,oixy->boxy', x_hat[:, :, :m1, :m2], self.W1[:, :, :m1, :m2])
        # bottom-left corner (negative kx)
        out_hat[:, :, -m1:, :m2] = torch.einsum(
            'bixy,oixy->boxy', x_hat[:, :, -m1:, :m2], self.W2[:, :, :m1, :m2])
        return torch.fft.irfft2(out_hat, s=(H, W))


class AAGELU(nn.Module):
    """Anti-aliased GELU. A pointwise nonlinearity injects high-frequency harmonics
    that ALIAS on a coarse grid, which is the main reason FNO-style nets are only
    approximately resolution-invariant. We upsample by `factor` (band-limited, via
    FFT zero-pad), apply GELU on the finer grid, then downsample (FFT truncate),
    which suppresses the aliased content. Operates on whatever (H,W) it receives,
    so it stays resolution-agnostic. Assumes even H,W (true for our grids)."""
    def __init__(self, factor=2):
        super().__init__()
        self.f = factor

    def forward(self, x):
        f = self.f
        if f == 1:
            return F.gelu(x)
        B, C, H, W = x.shape
        Xs = torch.fft.fftshift(torch.fft.fft2(x, dim=(-2, -1)), dim=(-2, -1))
        Hf, Wf = H * f, W * f
        ph, pw = (Hf - H) // 2, (Wf - W) // 2
        up = F.pad(Xs, (pw, Wf - W - pw, ph, Hf - H - ph))
        x_up = torch.fft.ifft2(torch.fft.ifftshift(up, dim=(-2, -1)), dim=(-2, -1)).real * (f * f)
        x_up = F.gelu(x_up)
        Ys = torch.fft.fftshift(torch.fft.fft2(x_up, dim=(-2, -1)), dim=(-2, -1))
        crop = Ys[..., ph:ph + H, pw:pw + W]
        return torch.fft.ifft2(torch.fft.ifftshift(crop, dim=(-2, -1)), dim=(-2, -1)).real / (f * f)


def _act(antialias):
    return AAGELU(2) if antialias else nn.GELU()


class FNO_Block2d(nn.Module):
    def __init__(self, width, modes1, modes2, antialias=False):
        super().__init__()
        self.conv = SpectralConv2d(width, width, modes1, modes2)
        self.skip = nn.Conv2d(width, width, 1)
        self.norm = nn.InstanceNorm2d(width)
        self.act = _act(antialias)

    def forward(self, x):
        return self.act(self.norm(self.conv(x) + self.skip(x)))


class FNO_Backbone2d(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, width=32, modes=16, n_layers=4, antialias=False):
        super().__init__()
        self.lift = nn.Conv2d(in_ch, width, 1)
        self.blocks = nn.ModuleList([FNO_Block2d(width, modes, modes, antialias=antialias)
                                     for _ in range(n_layers)])
        self.proj = nn.Sequential(
            nn.Conv2d(width, width, 1),
            _act(antialias),
            nn.Conv2d(width, out_ch, 1)
        )

    def forward(self, x):
        x = self.lift(x)
        for block in self.blocks:
            x = block(x)
        return self.proj(x)


class FunctionalNet2d(nn.Module):
    """FNO2d backbone → scalar functional F[u]. u:(B,1,H,W) → (B,)."""
    def __init__(self, width=24, modes=12, n_layers=3, antialias=False):
        super().__init__()
        self.backbone = FNO_Backbone2d(in_ch=1, out_ch=1, width=width,
                                        modes=modes, n_layers=n_layers, antialias=antialias)
        self.head = nn.Sequential(
            nn.Linear(1, 16),
            nn.GELU(),
            nn.Linear(16, 1)
        )

    def forward(self, u):
        density = self.backbone(u)              # (B,1,H,W)
        integral = density.mean(dim=(-1, -2))   # (B,1) — spatial average ∝ integral
        return self.head(integral).squeeze(-1)  # (B,)


# ============================================================================
# Model 1: Vanilla FNO (2D, residual)
# ============================================================================

class VanillaFNO2d(nn.Module):
    def __init__(self, width=32, modes=16, n_layers=4):
        super().__init__()
        self.backbone = FNO_Backbone2d(in_ch=1, out_ch=1, width=width,
                                        modes=modes, n_layers=n_layers)

    def forward(self, u):
        return u + self.backbone(u)

    def predict_with_info(self, u):
        return self.forward(u), {}


# ============================================================================
# Model 2: EP-FNO (2D, energy penalty)
# ============================================================================

class EP_FNO2d(nn.Module):
    def __init__(self, width=32, modes=16, n_layers=4):
        super().__init__()
        self.backbone = FNO_Backbone2d(in_ch=1, out_ch=1, width=width,
                                        modes=modes, n_layers=n_layers)

    def forward(self, u):
        return u + self.backbone(u)

    def predict_with_info(self, u):
        u_next = self.forward(u)
        E_in = 0.5 * (u**2).mean(dim=(-1, -2)).mean(dim=-1)
        E_out = 0.5 * (u_next**2).mean(dim=(-1, -2)).mean(dim=-1)
        return u_next, {'dE': E_out - E_in}

    def energy_penalty(self, info, pde_type):
        dE = info['dE']
        if pde_type in ('heat', 'burgers'):
            return (F.relu(dE)**2).mean()
        elif pde_type in ('wave', 'advection'):
            return (dE**2).mean()
        return torch.tensor(0.0, device=dE.device)


# ============================================================================
# Model 3: GENERIC-FNO (2D)
# ============================================================================

class GENERIC_FNO2d(nn.Module):
    """
    du/dt = L·δE/δu + M·δS/δu  with hard projection.
    L(kx,ky) = i·a  (anti-Hermitian diagonal), M(kx,ky) = |b|² (PSD diagonal).
    Operators are defined on the lowest (modes_op) modes in each direction,
    using two corners for +/- kx (rfft2 layout) → resolution invariant.
    """
    def __init__(self, nx=128, width_func=24, modes_func=12, n_layers_func=3,
                 modes_op=16, residual_gate_init=-3.0, l2_vargrad=False,
                 use_residual=True, degeneracy_construction=True,
                 antialias=False, integrator='euler'):
        super().__init__()
        self.nx = nx
        self.modes_op = modes_op
        # degeneracy_construction (DEFAULT, the thermodynamically-consistent model):
        #   build L = (I-P_S) D_L (I-P_S) and M = (I-P_E) D_M (I-P_E), where P_E,P_S
        #   are rank-1 projections onto delta E/delta u, delta S/delta u and D_L=i*a,
        #   D_M=|b|^2 are diagonal Fourier multipliers. Then L dS = 0 and M dE = 0
        #   EXACTLY, so energy is conserved (dE/dt=0) and entropy is produced
        #   (dS/dt=<dS,M dS> >= 0) by construction in ANY dimension -- no energy
        #   projection, no entropy correction, no free residual. A reversible PDE
        #   (wave) is forced to learn M->0 because dissipation can no longer hide
        #   behind a projection. Set False for the legacy projection+correction path
        #   (kept only for ablation; it does NOT specialize -- M and L are
        #   interchangeable under the projection, so everything routes through M).
        # l2_vargrad: use the L2 variational derivative (N/|Omega|) grad_u E. Affects
        #   only the operator-output SCALE (projections are scale-invariant ratios);
        #   harmless either way under the construction. Default off.
        # use_residual: only meaningful in the legacy path; a free residual would
        #   break the thermodynamic guarantee, so it is ignored when
        #   degeneracy_construction=True.
        self.degeneracy_construction = degeneracy_construction
        self.l2_vargrad = l2_vargrad
        self.use_residual = use_residual
        self.integrator = integrator   # 'euler' (default) or 'rk4' (norm-preserving)
        m1 = min(modes_op, nx // 2)
        m2 = min(modes_op, nx // 2 + 1)
        self.m1, self.m2 = m1, m2

        self.E_net = FunctionalNet2d(width=width_func, modes=modes_func,
                                     n_layers=n_layers_func, antialias=antialias)
        self.S_net = FunctionalNet2d(width=width_func, modes=modes_func,
                                     n_layers=n_layers_func, antialias=antialias)

        # L: anti-Hermitian diagonal, two kx corners
        self.a_pos = nn.Parameter(0.3 * torch.randn(m1, m2))
        self.a_neg = nn.Parameter(0.3 * torch.randn(m1, m2))
        # M: PSD diagonal, two kx corners (parameterized as |b|²)
        self.b_pos_r = nn.Parameter(0.3 * torch.randn(m1, m2))
        self.b_pos_i = nn.Parameter(0.3 * torch.randn(m1, m2))
        self.b_neg_r = nn.Parameter(0.3 * torch.randn(m1, m2))
        self.b_neg_i = nn.Parameter(0.3 * torch.randn(m1, m2))

        # Small gated residual for high-freq content
        self.residual = nn.Sequential(
            nn.Conv2d(1, 16, 1),
            nn.GELU(),
            nn.Conv2d(16, 1, 1)
        )
        self.residual_gate = nn.Parameter(torch.tensor(float(residual_gate_init)))

    def _apply_operators(self, dEdu_hat, dSdu_hat, H, W):
        """Apply diagonal L and M in 2D Fourier space (two kx corners)."""
        m1, m2 = self.m1, self.m2
        rev_hat = torch.zeros_like(dEdu_hat)
        diss_hat = torch.zeros_like(dSdu_hat)

        # L = i*a  (reversible)
        rev_hat[:, :, :m1, :m2] = 1j * self.a_pos * dEdu_hat[:, :, :m1, :m2]
        rev_hat[:, :, -m1:, :m2] = 1j * self.a_neg * dEdu_hat[:, :, -m1:, :m2]

        # M = |b|²  (dissipative, PSD)
        M_pos = self.b_pos_r**2 + self.b_pos_i**2
        M_neg = self.b_neg_r**2 + self.b_neg_i**2
        diss_hat[:, :, :m1, :m2] = M_pos * dSdu_hat[:, :, :m1, :m2]
        diss_hat[:, :, -m1:, :m2] = M_neg * dSdu_hat[:, :, -m1:, :m2]

        return rev_hat, diss_hat

    # --- single-operator Fourier multipliers (for degeneracy-by-construction) ---
    def _L_apply(self, v, H, W):
        """Apply the skew diagonal operator D_L = i*a to physical field v."""
        m1, m2 = self.m1, self.m2
        vh = torch.fft.rfft2(v, dim=(-2, -1))
        out = torch.zeros_like(vh)
        out[:, :, :m1, :m2] = 1j * self.a_pos * vh[:, :, :m1, :m2]
        out[:, :, -m1:, :m2] = 1j * self.a_neg * vh[:, :, -m1:, :m2]
        return torch.fft.irfft2(out, s=(H, W))

    def _M_apply(self, v, H, W):
        """Apply the PSD diagonal operator D_M = |b|^2 to physical field v."""
        m1, m2 = self.m1, self.m2
        vh = torch.fft.rfft2(v, dim=(-2, -1))
        out = torch.zeros_like(vh)
        Mp = self.b_pos_r**2 + self.b_pos_i**2
        Mn = self.b_neg_r**2 + self.b_neg_i**2
        out[:, :, :m1, :m2] = Mp * vh[:, :, :m1, :m2]
        out[:, :, -m1:, :m2] = Mn * vh[:, :, -m1:, :m2]
        return torch.fft.irfft2(out, s=(H, W))

    @staticmethod
    def _remove(v, w):
        """(I - P_w) v: remove the component of v along direction w, per sample.
        Scale-invariant in w (ratio), so the L2-vs-Euclidean choice is irrelevant."""
        ip = (v * w).sum(dim=(-1, -2), keepdim=True)
        nn = (w * w).sum(dim=(-1, -2), keepdim=True) + 1e-12
        return v - (ip / nn) * w

    def _generic_rhs(self, dEdu, dSdu, H, W):
        """du/dt = (I-P_S) D_L (I-P_S) dE + (I-P_E) D_M (I-P_E) dS.
        Degeneracy (L dS = 0, M dE = 0) holds exactly => dE/dt = 0 and
        dS/dt = <dS, M dS> >= 0 by construction, no projection needed."""
        rev = self._remove(self._L_apply(self._remove(dEdu, dSdu), H, W), dSdu)
        diss = self._remove(self._M_apply(self._remove(dSdu, dEdu), H, W), dEdu)
        return rev, diss

    def _field(self, u, H, W):
        """Reversible+dissipative increment rev+diss at state u (degeneracy path).
        u must require grad (RK4 intermediate states do). create_graph follows
        training so gradients flow through the integrator stages when training and
        eval stays memory-light."""
        cg = self.training
        E = self.E_net(u)
        S = self.S_net(u)
        dEdu = torch.autograd.grad(E.sum(), u, create_graph=cg)[0]
        dSdu = torch.autograd.grad(S.sum(), u, create_graph=cg)[0]
        if self.l2_vargrad:
            scale = (H * W) / (2.0 * math.pi) ** 2
            dEdu = dEdu * scale
            dSdu = dSdu * scale
        rev, diss = self._generic_rhs(dEdu, dSdu, H, W)
        return rev + diss

    def _rk4_step(self, u, H, W):
        """4th-order Runge-Kutta on the learned increment field. RK4's stability
        region contains a segment of the imaginary axis, so it suppresses the
        explicit-Euler amplitude growth of the skew (reversible) operator and
        tightens the finite-step energy drift to O(dt^5). Degeneracy
        (<dE,f>=0, <dS,Mf>>=0) holds at every stage, so the structural guarantees
        are unchanged; only the integration of them improves."""
        u0 = u.detach().requires_grad_(True)
        k1 = self._field(u0, H, W)
        k2 = self._field(u0 + 0.5 * k1, H, W)
        k3 = self._field(u0 + 0.5 * k2, H, W)
        k4 = self._field(u0 + k3, H, W)
        dudt = (k1 + 2.0 * k2 + 2.0 * k3 + k4) / 6.0
        return u + dudt

    def forward(self, u):
        B, C, H, W = u.shape

        if self.degeneracy_construction and self.integrator == 'rk4':
            return self._rk4_step(u, H, W)

        u_leaf = u.detach().requires_grad_(True)

        E = self.E_net(u_leaf)
        S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]

        if self.l2_vargrad:
            # L2 variational derivative: delta E/delta u = (N/|Omega|) grad_u E.
            scale = (H * W) / (2.0 * math.pi) ** 2
            dEdu = dEdu * scale
            dSdu = dSdu * scale

        if self.degeneracy_construction:
            # Thermodynamically-consistent path: degeneracy by construction (Euler).
            rev, diss = self._generic_rhs(dEdu, dSdu, H, W)
            dudt = rev + diss
            return u + dudt

        # --- legacy projection + correction path (ablation only) ---
        dEdu_hat = torch.fft.rfft2(dEdu, dim=(-2, -1))
        dSdu_hat = torch.fft.rfft2(dSdu, dim=(-2, -1))

        rev_hat, diss_hat = self._apply_operators(dEdu_hat, dSdu_hat, H, W)
        rev = torch.fft.irfft2(rev_hat, s=(H, W))
        diss = torch.fft.irfft2(diss_hat, s=(H, W))

        dudt = rev + diss
        dudt = self._project_energy_conservation(dudt, dEdu)
        dudt = self._ensure_entropy_production(dudt, dSdu, dEdu)

        if self.use_residual:
            gate = torch.sigmoid(self.residual_gate)
            residual = gate * self.residual(u_leaf)
            residual = self._project_energy_conservation(residual, dEdu)
            dudt = dudt + residual

        return u + dudt

    def _project_energy_conservation(self, dudt, dEdu):
        """Project du/dt ⊥ δE/δu over both spatial dims → dE/dt = 0."""
        inner = (dudt * dEdu).sum(dim=(-1, -2), keepdim=True)
        norm_sq = (dEdu * dEdu).sum(dim=(-1, -2), keepdim=True) + 1e-10
        return dudt - (inner / norm_sq) * dEdu

    def _ensure_entropy_production(self, dudt, dSdu, dEdu):
        """Ensure dS/dt = ⟨δS/δu, du/dt⟩ ≥ 0 via correction ⊥ δE/δu."""
        dSdt = (dSdu * dudt).sum(dim=(-1, -2), keepdim=True)
        violation = F.relu(-dSdt)
        if violation.sum() > 0:
            inner_SE = (dSdu * dEdu).sum(dim=(-1, -2), keepdim=True)
            norm_E_sq = (dEdu * dEdu).sum(dim=(-1, -2), keepdim=True) + 1e-10
            dSdu_perp = dSdu - (inner_SE / norm_E_sq) * dEdu
            inner_S_Sperp = (dSdu * dSdu_perp).sum(dim=(-1, -2), keepdim=True) + 1e-10
            alpha = violation / inner_S_Sperp
            dudt = dudt + alpha * dSdu_perp
        return dudt

    def entropy_production(self, u):
        """Normalized entropy production r_S = <dS/du, du/dt> / (||dS/du|| ||du/dt||),
        fully differentiable. Used as a minimum-entropy-production (MEP) penalty:
        among GENERIC representations consistent with the data, prefer the one that
        produces the least entropy. Scale-invariant in S (can't be gamed by
        rescaling S), so reducing it requires genuinely making du/dt more
        orthogonal to dS/du -- i.e. genuinely less dissipative."""
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf); S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]
        H, W = u.shape[-2], u.shape[-1]
        if self.l2_vargrad:
            scale = (H * W) / (2.0 * math.pi) ** 2
            dEdu = dEdu * scale
            dSdu = dSdu * scale
        if self.degeneracy_construction:
            rev, diss = self._generic_rhs(dEdu, dSdu, H, W)
            dudt = rev + diss
        else:
            dEh = torch.fft.rfft2(dEdu, dim=(-2, -1))
            dSh = torch.fft.rfft2(dSdu, dim=(-2, -1))
            rev_hat, diss_hat = self._apply_operators(dEh, dSh, H, W)
            rev = torch.fft.irfft2(rev_hat, s=(H, W))
            diss = torch.fft.irfft2(diss_hat, s=(H, W))
            dudt = rev + diss
            dudt = self._project_energy_conservation(dudt, dEdu)
            dudt = self._ensure_entropy_production(dudt, dSdu, dEdu)
            if self.use_residual:
                gate = torch.sigmoid(self.residual_gate)
                res = self._project_energy_conservation(gate * self.residual(u_leaf), dEdu)
                dudt = dudt + res
        num = (dSdu * dudt).flatten(1).sum(dim=1)
        den = dSdu.flatten(1).norm(dim=1) * dudt.flatten(1).norm(dim=1) + 1e-8
        return (num / den).clamp(min=0).mean()

    def predict_with_info(self, u):
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf)
        S = self.S_net(u_leaf)
        _ = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        _ = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]
        u_next = self.forward(u)
        u_next_leaf = u_next.detach().requires_grad_(True)
        E_next = self.E_net(u_next_leaf)
        S_next = self.S_net(u_next_leaf)
        info = {
            'E': E.detach(), 'S': S.detach(),
            'dE': (E_next - E).detach(), 'dS': (S_next - S).detach(),
        }
        return u_next, info

    def degeneracy_loss(self, u):
        """Soft regularizer: L·δS/δu ≈ 0 and M·δE/δu ≈ 0.
        Under degeneracy_construction these hold exactly, so the penalty is 0
        (kept only for the legacy projection path)."""
        if self.degeneracy_construction:
            return torch.zeros((), device=u.device)
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf)
        S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]
        dEdu_hat = torch.fft.rfft2(dEdu, dim=(-2, -1))
        dSdu_hat = torch.fft.rfft2(dSdu, dim=(-2, -1))
        m1, m2 = self.m1, self.m2

        # L·δS/δu
        L_dS_pos = 1j * self.a_pos * dSdu_hat[:, :, :m1, :m2]
        L_dS_neg = 1j * self.a_neg * dSdu_hat[:, :, -m1:, :m2]
        loss_L = (L_dS_pos.abs()**2).mean() + (L_dS_neg.abs()**2).mean()

        # M·δE/δu
        M_pos = self.b_pos_r**2 + self.b_pos_i**2
        M_neg = self.b_neg_r**2 + self.b_neg_i**2
        M_dE_pos = M_pos * dEdu_hat[:, :, :m1, :m2]
        M_dE_neg = M_neg * dEdu_hat[:, :, -m1:, :m2]
        loss_M = (M_dE_pos.abs()**2).mean() + (M_dE_neg.abs()**2).mean()

        return loss_L + loss_M


# ============================================================================
# Training
# ============================================================================


# ============================================================================
# Evaluation
# ============================================================================



# ==================== PART C: COMPUTE & MEMORY BENCHMARK ======================
# Measures the overhead of GENERIC-FNO's autodiff-through-functionals forward
# (create_graph=True) against a plain FNO and an energy-penalized FNO, at the
# paper's 2D config. Reports params, forward latency, train-step latency
# (fwd+bwd+opt), and peak GPU memory, plus overhead multipliers vs FNO, and a
# ready-to-paste LaTeX table for Appendix E.
import os, time
import numpy as np
import torch

BM_RES    = int(os.environ.get("BM_RES", 128))    # paper headline grid
BM_BATCH  = int(os.environ.get("BM_BATCH", 8))    # lower if OOM; ratios are ~batch-stable
BM_WARMUP = int(os.environ.get("BM_WARMUP", 5))
BM_ITERS  = int(os.environ.get("BM_ITERS", 30))
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"

# paper 2D sizes
_WIDTH, _MODES, _NLAY = 32, 16, 4
_WF, _MF, _NLF = 24, 12, 3

def _cp(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
def _is_gen(m): return hasattr(m, "E_net")
def _sync():
    if DEVICE == "cuda": torch.cuda.synchronize()

def _build():
    return {
        "FNO":         VanillaFNO2d(_WIDTH, _MODES, _NLAY),
        "EP-FNO":      EP_FNO2d(_WIDTH, _MODES, _NLAY),
        "GENERIC-FNO": GENERIC_FNO2d(BM_RES, _WF, _MF, _NLF, _MODES),
    }

def _rel(pred, tgt):
    p = pred.reshape(pred.shape[0], -1); t = tgt.reshape(tgt.shape[0], -1)
    return (torch.linalg.vector_norm(p - t, dim=1) /
            (torch.linalg.vector_norm(t, dim=1) + 1e-12)).mean()

def _time_forward(model, u):
    """Train-mode forward latency (ms/step): GENERIC builds the create_graph=True
    graph it needs for the double-backward, so this is the pessimistic forward."""
    gen = _is_gen(model); model.eval()
    def once():
        if gen: return model(u)
        with torch.no_grad(): return model(u)
    for _ in range(BM_WARMUP): once()
    _sync(); t0 = time.perf_counter()
    for _ in range(BM_ITERS): once()
    _sync(); return (time.perf_counter() - t0) / BM_ITERS * 1e3

def _time_forward_deploy(model, u):
    """Deployed inference latency (ms/step). For GENERIC we force
    create_graph=False (no double-backward is needed at inference), which is the
    honest deployment cost; FNO/EP-FNO already run under no_grad in _time_forward."""
    if not _is_gen(model):
        return _time_forward(model, u)
    model.eval()
    orig = torch.autograd.grad
    def patched(*a, **k):
        k["create_graph"] = False
        return orig(*a, **k)
    torch.autograd.grad = patched
    try:
        def once(): return model(u)
        for _ in range(BM_WARMUP): once()
        _sync(); t0 = time.perf_counter()
        for _ in range(BM_ITERS): once()
        _sync(); return (time.perf_counter() - t0) / BM_ITERS * 1e3
    finally:
        torch.autograd.grad = orig

def _time_train_step(model, u, tgt):
    """Full training step latency (ms): forward + data-loss backward + opt.step.
    For GENERIC the backward traverses the create_graph=True graph."""
    model.train(); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    def step():
        opt.zero_grad(); loss = _rel(model(u), tgt); loss.backward(); opt.step()
    for _ in range(BM_WARMUP): step()
    _sync(); t0 = time.perf_counter()
    for _ in range(BM_ITERS): step()
    _sync(); return (time.perf_counter() - t0) / BM_ITERS * 1e3

def _peak_mem_train(model, u, tgt):
    """Peak GPU memory (MB) during one training step (params+optimizer+activations)."""
    if DEVICE != "cuda": return float("nan")
    model.train(); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    opt.zero_grad(); _rel(model(u), tgt).backward(); opt.step()   # warm optimizer state
    _sync(); torch.cuda.reset_peak_memory_stats()
    opt.zero_grad(); _rel(model(u), tgt).backward(); opt.step()
    _sync(); return torch.cuda.max_memory_allocated() / 1e6

def run_compute_benchmark(seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    print(f"Compute benchmark  res={BM_RES} batch={BM_BATCH} device={DEVICE} "
          f"(warmup={BM_WARMUP}, iters={BM_ITERS})")
    u   = torch.randn(BM_BATCH, 1, BM_RES, BM_RES, device=DEVICE)
    tgt = torch.randn(BM_BATCH, 1, BM_RES, BM_RES, device=DEVICE)
    rows = {}
    for name, model in _build().items():
        model = model.to(DEVICE)
        try:
            p  = _cp(model)
            tf = _time_forward(model, u)          # train-mode forward (create_graph=True for GENERIC)
            td = _time_forward_deploy(model, u)    # deployed inference (create_graph=False for GENERIC)
            tt = _time_train_step(model, u, tgt)
            mm = _peak_mem_train(model, u, tgt)
            rows[name] = dict(params=p, fwd=tf, fwd_deploy=td, train=tt, mem=mm)
            mems = f"{mm:.0f}MB" if mm == mm else "n/a(cpu)"
            print(f"  {name:12s}: params={p:,}  fwd_deploy={td:.1f}ms  fwd_train={tf:.1f}ms  "
                  f"train_step={tt:.1f}ms  peak_mem={mems}")
        except RuntimeError as e:
            print(f"  {name:12s}: FAILED ({str(e)[:80]}). Try a smaller BM_BATCH.")
        finally:
            del model
            if DEVICE == "cuda": torch.cuda.empty_cache()
    _report(rows)
    return rows

def _report(rows):
    if "FNO" not in rows:
        print("  [WARN] FNO row missing; skipping ratios."); return
    base = rows["FNO"]
    print("\n==== overhead relative to plain FNO ====")
    for name, r in rows.items():
        fxd = r["fwd_deploy"] / base["fwd_deploy"]
        fxt = r["fwd"] / base["fwd"]
        tx = r["train"] / base["train"]
        mx = (r["mem"] / base["mem"]) if (r["mem"] == r["mem"] and base["mem"] == base["mem"]) else float("nan")
        pxp = r["params"] / base["params"]
        mxs = f"{mx:.2f}x" if mx == mx else "n/a"
        print(f"  {name:12s}: params {pxp:.2f}x  fwd_deploy {fxd:.2f}x  fwd_train {fxt:.2f}x  "
              f"train_step {tx:.2f}x  peak_mem {mxs}")
    # LaTeX (Appendix E) -- table uses the deployed-inference forward
    def mem(r): return f"{r['mem']:.0f}" if r['mem'] == r['mem'] else "--"
    print("\n% ---- LaTeX (Appendix E, computational cost) ----")
    print(r"\begin{tabular}{lcccc}")
    print(r"\toprule")
    print(r"Model & Params & Forward (ms) & Train step (ms) & Peak mem (MB) \\")
    print(r"\midrule")
    for name in ["FNO", "EP-FNO", "GENERIC-FNO"]:
        if name not in rows: continue
        r = rows[name]
        print(f"{name} & {r['params']:,} & {r['fwd_deploy']:.1f} & {r['train']:.1f} & {mem(r)} \\\\".replace(",", "{,}"))
    print(r"\bottomrule")
    print(r"\end{tabular}")
    tm = ", ".join(f"{n} {rows[n]['fwd']:.1f}ms" for n in ["FNO","EP-FNO","GENERIC-FNO"] if n in rows)
    print("% train-mode forward (create_graph=True), for reference: " + tm)

if __name__ == "__main__":
    run_compute_benchmark()

Compute benchmark  res=128 batch=8 device=cuda (warmup=5, iters=30)
  FNO         : params=2,102,529  fwd_deploy=3.0ms  fwd_train=2.8ms  train_step=9.6ms  peak_mem=358MB
  EP-FNO      : params=2,102,529  fwd_deploy=2.8ms  fwd_train=2.8ms  train_step=9.8ms  peak_mem=392MB
  GENERIC-FNO : params=1,001,958  fwd_deploy=13.0ms  fwd_train=14.0ms  train_step=98.6ms  peak_mem=809MB

==== overhead relative to plain FNO ====
  FNO         : params 1.00x  fwd_deploy 1.00x  fwd_train 1.00x  train_step 1.00x  peak_mem 1.00x
  EP-FNO      : params 1.00x  fwd_deploy 0.95x  fwd_train 0.99x  train_step 1.02x  peak_mem 1.09x
  GENERIC-FNO : params 0.48x  fwd_deploy 4.41x  fwd_train 5.00x  train_step 10.25x  peak_mem 2.26x

% ---- LaTeX (Appendix E, computational cost) ----
\begin{tabular}{lcccc}
\toprule
Model & Params & Forward (ms) & Train step (ms) & Peak mem (MB) \\
\midrule
FNO & 2{,}102{,}529 & 3.0 & 9.6 & 358 \\
EP-FNO & 2{,}102{,}529 & 2.8 & 9.8 & 392 \\
GENERIC-FNO & 1{,}001{,}958 & 13.0 & 98.6